# Colab용 영화 리뷰 재분석기 (Ollama + Gemma4)
이 노트북은 `tagged_reviews.json` 파일에서 룰베이스(Rule-based)로 임시 태깅된 리뷰들만 선별하여 다시 강력한 옵션과 함께 LLM을 통해 정교한 태깅을 재시도합니다.

In [ ]:
# 1. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Ollama 설치 및 백그라운드 실행, Gemma4 모델 다운로드
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

# 서버 실행
process = subprocess.Popen(['ollama', 'serve'])
time.sleep(3)

# 파이썬용 ollama 패키지 설치
!pip install ollama

# 모델 다운로드
!ollama pull gemma4

In [ ]:
# 3. 라이브러리 임포트 및 경로 설정
import json
import os
import time
import re
import ollama

# 🏗️ 코랩 환경에서의 프로젝트 루트 경로 설정
ROOT_DIR = "/content/drive/MyDrive/movieReviews"
DATA_DIR = os.path.join(ROOT_DIR, "data")

LOCAL_MODEL = "gemma4"

# 🚀 테스트 모드 스위치
IS_TEST_MODE = False  # True: 일부 리뷰만 분석 / False: 전체 분석
TEST_LIMIT = 20

print(f"데이터 경로 설정 완료: {DATA_DIR}")

In [ ]:
# 4. LLM 함수 및 메인 재분석 로직 정의

def classify_batch_with_local_llm(review_batch, max_retries=5):
    batch_text = ""
    for idx, r in enumerate(review_batch):
        batch_text += f"ID:{idx} | 본문: {r['content']}\n"

    prompt = f"""
    당신은 영화 리뷰 분석 전문가입니다. 다음 {len(review_batch)}개의 리뷰를 각각 분석하여 정보를 추출하세요.
    응답은 반드시 한국어로 작성하세요.

    [범주]: 전체긍정, 전체부정, 전체복합, 연기좋음, 연기나쁨, 연출좋음, 연출나쁨, 서사좋음, 서사나쁨, 비주얼좋음, 비주얼나쁨, 음악좋음, 분위기가벼움, 분위기무거움, 고증좋음, 고증나쁨, 주의사항, TMI, 장르특성
    
    [분석 지침]:
    - 각 리뷰의 본문을 꼼꼼히 읽고 해당되는 모든 범주를 선택하세요.
    - 가장 먼저 리뷰의 전체적인 총평이나 분위기가 긍정적이면 '전체긍정', 부정적이면 '전체부정', 장단점이 섞여있거나 모호하면 '전체복합' 중 하나를 반드시 포함하세요.
    - 특히 '연기', '연출', '서사', '비주얼', '음악', '분위기'는 단순히 내용 언급이 아니라, 긍정적인 평가면 '좋음', 부정적인 평가면 '나쁨'을 붙여서 상세히 태깅하세요.
    - ⚠️ 절대 비워두지 마세요: 모든 리뷰에 대해 반드시 최소 1개 이상의 'content_character'를 찾아야 합니다.
    - ⚠️ 절대 비워두지 마세요: 'search_keywords'는 본문 내용에서 유추할 수 있는 대표 핵심 단어를 개수 제한 없이 최대한 다양하고 풍부하게 추출하세요 (최소 1개 이상).
    
    [응답 지시 (매우 중요)]:
    1. 반드시 아래 제시된 JSON 리스트 형식으로만 응답해야 합니다. 다른 부가적인 텍스트나 마크다운 코드 블록(```json 등)은 절대 포함하지 마세요.
    2. JSON 문법을 완벽하게 지켜야 합니다. 키와 값은 반드시 큰따옴표(")로 감싸야 하며, 문자열 내부에 큰따옴표가 들어갈 경우 작은따옴표(')로 대체하세요.
    3. 항목 사이의 쉼표(,)를 절대 누락하지 마세요.
    4. 요청받은 {len(review_batch)}개의 리뷰에 대한 분석 결과가 모두 리스트 안에 포함되어야 합니다. 중간에 생략하지 마세요.

    형식 예시 (실제 값으로 대체할 것):
    [
        {{"id": 0, "content_character": ["추출한태그1", "추출한태그2"], "search_keywords": ["핵심단어1", "핵심단어2", "핵심단어3"]}},
        {{"id": 1, "content_character": ["추출한태그3"], "search_keywords": ["핵심단어4"]}}
    ]

    [리뷰 리스트]:
    {batch_text}
    """

    for attempt in range(max_retries):
        try:
            response = ollama.chat(model=LOCAL_MODEL, messages=[
                {'role': 'user', 'content': prompt},
            ])
            
            response_text = response['message']['content']
            json_text = re.sub(r'```json|```', '', response_text).strip()
            results = json.loads(json_text)

            if len(results) != len(review_batch):
                raise ValueError(f"응답 개수 불일치 (요청: {len(review_batch)}개, 응답: {len(results)}개)")

            for res in results:
                content_chars = res.get("content_character", [])
                keywords = res.get("search_keywords", [])
                if not content_chars or not keywords:
                    raise ValueError(f"ID {res.get('id')}의 태그나 키워드가 비어있습니다.")
                if "추출한태그1" in content_chars or "핵심단어1" in keywords:
                    raise ValueError(f"ID {res.get('id')}가 예시를 그대로 복사했습니다.")
                
                original_review = next((r for idx, r in enumerate(review_batch) if idx == res['id']), None)
                if original_review and len(original_review['content']) > 100:
                    if len(content_chars) < 2:
                        raise ValueError(f"ID {res.get('id')}는 100자 이상 리뷰인데 태그가 부족합니다.")

            return sorted(results, key=lambda x: x['id'])
        except Exception as e:
            print(f"⚠️ 로컬 LLM 분석 오류 (시도 {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                print("❌ 최대 재시도 횟수 초과. Fallback(Rule-based) 유지.")
                return None

# 5. 메인 실행 블록
target_path = os.path.join(DATA_DIR, "tagged_reviews.json")

if not os.path.exists(target_path):
    print(f"❌ '{target_path}' 파일이 없습니다. 먼저 classifier_colab.ipynb 를 통해 전체 리뷰 분석을 진행해주세요.")
else:
    with open(target_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    pending_reviews = []
    pending_refs = [] 

    # 룰베이스(Rule-based)로 태깅된 리뷰만 필터링
    for m_name, m_data in data.items():
        for i, rev in enumerate(m_data.get("reviews", [])):
            method = rev.get("meta_tags", {}).get("analysis_method", "")
            if method == "Rule-based (Fallback)":
                pending_reviews.append(rev)
                pending_refs.append((m_name, i))

    if not pending_reviews:
        print("✅ 룰베이스로 분류된 리뷰가 없습니다! 모두 정상적으로 LLM으로 분석되었습니다.")
    else:
        if IS_TEST_MODE:
            pending_reviews = pending_reviews[:TEST_LIMIT]
            pending_refs = pending_refs[:TEST_LIMIT]
            print(f"⚠️ [테스트 모드 활성화] 리뷰 수를 {len(pending_reviews)}개로 제한합니다.")

        print(f"🚀 총 {len(pending_reviews)}개의 룰베이스 처리된 리뷰를 재분석합니다.")

        batch_size = 10 
        success_count = 0
        fail_count = 0

        for i in range(0, len(pending_reviews), batch_size):
            current_batch = pending_reviews[i:i+batch_size]
            current_refs = pending_refs[i:i+batch_size]
            print(f"📦 재분석 중... ({i}/{len(pending_reviews)})")

            results = classify_batch_with_local_llm(current_batch)

            if results and len(results) == len(current_batch):
                for idx, res in enumerate(results):
                    m_name, rev_idx = current_refs[idx]
                    
                    data[m_name]["reviews"][rev_idx]["meta_tags"].update({
                        "content_character": res.get("content_character", []),
                        "search_keywords": res.get("search_keywords", []),
                        "is_tmi": "TMI" in res.get("content_character", []),
                        "has_warning": "주의사항" in res.get("content_character", []),
                        "analysis_method": "Local-LLM"
                    })
                    success_count += 1
            else:
                print("⚠️ 배치 재분석 실패. 기존 룰베이스를 유지합니다.")
                fail_count += len(current_batch)

            if (i // batch_size) % 3 == 0:
                with open(target_path, "w", encoding="utf-8") as f:
                    json.dump(data, f, ensure_ascii=False, indent=4)

        with open(target_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=4)

        print(f"\n✨ 재분석 완료! (성공: {success_count}개 / 실패 및 유지: {fail_count}개)")
        print(f"결과 저장: {target_path}")
